# Final emotion data and loader

Final version: Sad=0, Happy=1, Neutral=2. The summary cell below was executed locally from the saved audit record. The archive-verification and batch cells are runnable instructions, not a new claimed data audit. No source images are embedded in this notebook.

In [1]:
import json

summary = json.loads(
    '{"version": "sad-happy-neutral-geometry-v2-20261121", "review_date": "2026-11-21", "source_dataset": "FER2013", "source_url": "https://www.kaggle.com/datasets/msambare/fer2013", "parent_manifest_sha256": "ce7e0c90e6b44243392587c135bad92542a050c66c9f38f1d6311643669b9429", "review_decisions_sha256": "9ad839ddee49a3b363845b5a71f6a0a130fe5d1c357f611fe8837ec2644b8cab", "manifest_sha256": "54ac9d2ea037dbb2d879acd9d960f11161bfc353b89f5ca8a0151f221d2ac4ec", "parent_images": 20716, "retained_images": 20253, "excluded_images": 463, "excluded_by_reason": {"extreme_profile": 284, "severe_crop": 179}, "retained_counts": {"train": {"sad": 3581, "happy": 5541, "neutral": 3781}, "val": {"sad": 890, "happy": 1384, "neutral": 943}, "test": {"sad": 1186, "happy": 1749, "neutral": 1198}}, "excluded_counts": {"train/sad": 144, "test/sad": 49, "train/happy": 73, "test/neutral": 26, "test/happy": 17, "train/neutral": 66, "val/happy": 22, "val/neutral": 21, "val/sad": 45}, "model_ids": {"sad": 0, "happy": 1, "neutral": 2}, "labels_changed": 0, "split_membership_changed": 0, "policy": {"severe_crop": "Image boundary removes an eye or the mouth region; ordinary tight forehead/chin crops retained.", "extreme_profile": "Head rotation hides one eye and much of the far side of the face; slight turns and borderline cases retained."}, "review_method": "AI-assisted visual screening of all 20,716 images on 41 shuffled contact sheets; 4,415 candidates reviewed on 35 enlarged sheets; all proposed exclusions checked again. Class, split and model predictions hidden on review sheets. Saved visual decisions fix the exact package membership.", "limitations": "Subjective visual review at original 48x48 resolution; may miss cases. This is a geometry audit, not a complete content or expression-label audit. Unrelated content flags retained separately. No independent reviewer confirmation yet.", "evaluation": "No training or model inference performed. Previous metrics belong to the parent dataset. Evaluate any new model separately; do not transfer earlier results to this version.", "verification": "All retained image bytes match parent SHA256; original grayscale 48x48 images; one copy per decoded-pixel hash across all splits; file inventory and ZIP contents verified."}'
)
print(
    json.dumps(
        {
            k: summary[k]
            for k in [
                "version",
                "parent_images",
                "retained_images",
                "excluded_by_reason",
                "retained_counts",
                "model_ids",
                "manifest_sha256",
            ]
        },
        indent=2,
    )
)

{
  "version": "sad-happy-neutral-geometry-v2-20261121",
  "parent_images": 20716,
  "retained_images": 20253,
  "excluded_by_reason": {
    "extreme_profile": 284,
    "severe_crop": 179
  },
  "retained_counts": {
    "train": {
      "sad": 3581,
      "happy": 5541,
      "neutral": 3781
    },
    "val": {
      "sad": 890,
      "happy": 1384,
      "neutral": 943
    },
    "test": {
      "sad": 1186,
      "happy": 1749,
      "neutral": 1198
    }
  },
  "model_ids": {
    "sad": 0,
    "happy": 1,
    "neutral": 2
  },
  "manifest_sha256": "54ac9d2ea037dbb2d879acd9d960f11161bfc353b89f5ca8a0151f221d2ac4ec"
}


## Load the fixed extracted dataset
Set EMOTION_DATA_ROOT to extracted files, or use the verified Drive ZIP as shown in emotion_final_training.ipynb. No new split or relabeling occurs. The loader validates byte hashes and split separation; it does not establish correct expression annotations.

In [ ]:
import csv
import os
import sys
from pathlib import Path

repo = Path.cwd() if (Path.cwd() / "scripts").is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo / "scripts" / "emotion_final"))
from load_cleaned_zip import validate_dataset  # noqa: E402 — import from the selected source checkout.

data = Path(os.environ["EMOTION_DATA_ROOT"])
info = validate_dataset(data)
with (data / "manifest.csv").open(encoding="utf-8", newline="") as f:
    rows = [r for r in csv.DictReader(f) if r["status"] == "retained"]
print(info["retained_counts"])

In [ ]:
import torch
from PIL import Image
from torchvision import transforms

transform = transforms.Compose(
    [
        transforms.Grayscale(3),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)
flip = transforms.RandomHorizontalFlip(0.5)
for split in ["train", "val", "test"]:
    selected = [r for r in rows if r["output_split"] == split][:32]
    images = []
    for r in selected:
        with Image.open(data / r["output_path"]) as image:
            images.append(transform(flip(image) if split == "train" else image))
    x = torch.stack(images)
    y = torch.tensor([int(r["model_id"]) for r in selected], dtype=torch.int64)
    print(split, x.shape, x.dtype, y.shape, y.dtype)